In [ ]:
#Package loading
import pickle
import os
import json
import math
import torch
import random
import hashlib
import argparse

import numpy as np
import pandas as pd
from tqdm import tqdm
import torch.nn as nn
from copy import deepcopy
import scipy.sparse as sp
import torch.optim as optim
import torch.nn.functional as F
import torch.optim.lr_scheduler as lr_scheduler

from scipy.sparse import coo_matrix
from collections import defaultdict
from torch_scatter import scatter_sum
from torch_geometric.utils import softmax
from torch_geometric.data import Data, Batch
from lifelines.utils import concordance_index
from torch.utils.data import Dataset,DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from torch_geometric.utils import dense_to_sparse
from torch_geometric.nn import GCNConv, GATConv,GraphNorm,global_mean_pool
from sklearn.metrics import roc_auc_score,average_precision_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef
)

from utils import *
import warnings
warnings.filterwarnings("ignore")

device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
print('torch version: ', torch.__version__)
torch.cuda.set_device(1)


In [ ]:
#model
class PathwayAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(dim, dim),
            nn.Tanh(),
            nn.Linear(dim, 1)
        )

    def forward(self, x, batch):
        """
        Args:
            x: [Total_Genes_in_Batch, dim]
            batch: [Total_Genes_in_Batch]
        """
        # Calculate the original score for each gene
        raw_attn = self.gate(x)          # [Total_Genes_in_Batch, 1]
        
        # Ensure that the sum of the gene weights within the pathway is 1, and that different pathways do not affect each other
        attn = softmax(raw_attn, batch)  # [Total_Genes_in_Batch, 1]
        
        weighted_x = x * attn            # [Total_Genes_in_Batch, dim]
        
        # Aggregate to obtain pathway representation
        pathway_feats = scatter_sum(weighted_x, batch, dim=0) 
        
        return pathway_feats,attn
 
class InterpretableTransformerLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_ff=128, dropout=0.0):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(   # Initialization of multi-head attention
            dropout=dropout,
            embed_dim=d_model,
            num_heads=nhead,
            batch_first=True
        )
        self.ffn = nn.Sequential(                  # FNN
            nn.Linear(d_model, dim_ff),
            nn.ReLU(),
            nn.Linear(dim_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x_norm = self.norm1(x)
        attn_out, attn_weights = self.self_attn(
            x_norm, x_norm, x_norm,
            need_weights=True,
            average_attn_weights=False
        )
        x = x + self.dropout(attn_out)
        x_norm = self.norm2(x)
        ffn_out = self.ffn(x_norm)
        x = x + self.dropout(ffn_out)
        return x, attn_weights

class TwoLForwardNetwork(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim1, hidden_dim2, dropout_pathway = 0.1,
                 num_class = 5,activation_func=nn.ReLU(),out_activation=None):
        super(TwoLForwardNetwork, self).__init__()
        
        self.layers = nn.Sequential(
             nn.Linear(input_dim, hidden_dim1).to(device),
             nn.BatchNorm1d(hidden_dim1),
             activation_func,
             nn.Dropout(dropout_pathway),
             nn.Linear(hidden_dim1, hidden_dim2).to(device),
             nn.BatchNorm1d(hidden_dim2),
             activation_func,
             nn.Dropout(dropout_pathway),
             nn.Linear(hidden_dim2, num_class)).to(device)
        
        self.out_activation = out_activation

    def forward(self, x):
        if self.out_activation:
            return self.out_activation(self.layers(x))
        else:
            return self.layers(x)

class GenePathwayProcessor:
    def __init__(self, pathways: dict):
        # Preprocessing pathway
        self.pathway_info = self._preprocess_pathways(pathways)
        
        # Construct a global gene index
        all_genes = list(set(g for p in self.pathway_info for g in p["genes"]))
        self.gene2idx = {g: i for i, g in enumerate(all_genes)}
        self.num_genes = len(all_genes)

    def _preprocess_pathways(self, pathways: dict):
        
        pathway_list = []
        for p_name, adj_df in pathways.items():
            
            adj_matrix = adj_df.values  
            edge_index, _ = dense_to_sparse(torch.tensor(adj_matrix, dtype=torch.float32))
            pathway_genes = adj_df.index.tolist()
            pathway_list.append({
                "name": p_name,
                "edge_index": edge_index,  # [2, E],E is the number of edges
                "genes": pathway_genes,    
                "num_genes": len(pathway_genes)
            })
        return pathway_list

    def process_batch(self, expr_df: pd.DataFrame, sample_names: list):

        num_samples = len(sample_names)
        
        # Construct sample × gene expression matrix (fill in 0 for missing values)
        expr_matrix = expr_df.reindex(self.gene2idx.keys())[sample_names].fillna(0.0).T
        expr_tensor = torch.tensor(expr_matrix.values, dtype=torch.float32)  # [N_samples, N_genes]
        
  
        # Pathway network data were constructed for each sample
   
        batch_list = []
        for sample_idx in range(num_samples):
            sample_data_list = []
            sample_expr = expr_tensor[sample_idx]
            
            for p in self.pathway_info:
                gene_indices = torch.tensor([self.gene2idx[g] for g in p["genes"]], dtype=torch.long)
                # [num_genes_in_pathway, 1]）
                x = sample_expr[gene_indices].unsqueeze(1)  
                
                data = Data(
                    x=x,
                    edge_index=p["edge_index"],  
                    num_nodes=p["num_genes"]
                )
                sample_data_list.append(data)
            
            sample_batch = Batch.from_data_list(sample_data_list)
            batch_list.append(sample_batch)
        

        #Combine the batches of all samples 
        total_batch = Batch.from_data_list(batch_list)
        return total_batch

class PathwayGATModel(torch.nn.Module):
    def __init__(self,
                 gene_in_dim: int = 1,             #The input dimension of genes
                 gene_hidden_dim1: int = 8,        #The output dimension of the first GCN layer
                 gene_hidden_dim2: int = 32,       #The output dimension of the second GCN layer
                 pathway_in_dim: int = 32,         #The input dimension of the GAT layer
                 pathway_hidden_dim1: int = 8,     #The output dimension of the GAT layer
                 MLP_input_dim: int = 2504,        #Input dimension of the prediction layer (multi-layer perceptron)
                 MLP_hidden_dim1: int = 512,
                 MLP_hidden_dim2: int = 64 , 
                 num_class: int = 5,               #Number of prediction categories
                 dropout_pathway: float = 0.5,
                 GAT_dropout: float = 0.5,
                 transformer_heads: int = 2,
                 transformer_layers: int = 3,
                 final_pathway_dim: int =32):
        super(PathwayGATModel, self).__init__()


        # Intra-pathway Topology Encoding
        
        self.GCN1 = GCNConv(gene_in_dim, gene_hidden_dim1,add_self_loops=True)
        self.GCN2 = GCNConv(gene_hidden_dim1, gene_hidden_dim2,add_self_loops=True)
        self.gene_pool1 = PathwayAttention(gene_hidden_dim1)
        self.gene_pool2 = PathwayAttention(gene_hidden_dim2)
        
        # Local Pathway Crosstalk Encoding
        
        self.pathway_GAT1 = GATConv(in_channels=pathway_in_dim,
                                    out_channels=pathway_hidden_dim1,
                                    heads=3,concat = False,dropout=GAT_dropout,
                                    add_self_loops= True,edge_dim=1)

        # Global pathway dependency
        
        self.transformer_layers = nn.ModuleList([
            InterpretableTransformerLayer(
                dim_ff = 512,
                d_model = pathway_hidden_dim1,
                nhead = transformer_heads,
                dropout = 0.3
            ) for _ in range(transformer_layers)
        ])

        # MLP

        self.net = TwoLForwardNetwork(MLP_input_dim,
                                      MLP_hidden_dim1,
                                      MLP_hidden_dim2,
                                      dropout_pathway,num_class,
                                      activation_func = nn.GELU())
        
        self.activation = nn.GELU()
        self.gene_norm1 = GraphNorm(gene_hidden_dim1)
        self.gene_norm2 = GraphNorm(gene_hidden_dim2)
        self.pre_transformer_proj = nn.Linear(pathway_hidden_dim1, pathway_hidden_dim1)
        self.pre_transformer_ln = nn.LayerNorm(pathway_hidden_dim1)
        self.token_dim_reduction = nn.Linear(gene_hidden_dim1+gene_hidden_dim2,final_pathway_dim)
        
    def forward(self,
            expr_df: pd.DataFrame,                         #Expression Matrix (Gene * Sample)
            sample_names: list,                            #Pathway-Pathway Adjacency matrix (Batch)
            pathway_edge_index1: torch.Tensor,             
            pathway_edge_index2: torch.Tensor,             #Similarity between pathways
            pathway_edge_weight1: torch.Tensor,            
            pathway_edge_weight2: torch.Tensor,
            batch_size: int) -> torch.Tensor:              #Batch Size

        device = next(self.parameters()).device
        batch_res = processor.process_batch(expr_df, sample_names).to(device)

        gene_hidden1 = self.GCN1(batch_res.x, batch_res.edge_index)
        gene_hidden1 = self.gene_norm1(gene_hidden1, batch_res.batch)
        gene_hidden1 = self.activation(gene_hidden1)
        pathway_feats1,gene_atte1 = self.gene_pool1(gene_hidden1,batch_res.batch)
        
        gene_hidden2 = self.GCN2(gene_hidden1, batch_res.edge_index)
        gene_hidden2 = self.gene_norm2(gene_hidden2, batch_res.batch)
        gene_hidden2 = self.activation(gene_hidden2)
        pathway_feats2,gene_atte2 = self.gene_pool2(gene_hidden2,batch_res.batch)
        
        
        pathway_feats = torch.cat([pathway_feats1,pathway_feats2],dim=1)
        
        num_samples = len(sample_names)
        
        if num_samples == batch_size: 
            edge_index_b = pathway_edge_index1
            edge_weight_b = pathway_edge_weight1  
        else:       
            edge_index_b = pathway_edge_index2 
            edge_weight_b = pathway_edge_weight2 
            
        
        p_emb1, (edge_idx_attn, gat_attn) = self.pathway_GAT1(
            pathway_feats,
            edge_index_b,
            edge_weight_b,
            return_attention_weights=True
        )
        p_emb1 = self.activation(p_emb1)

        # Residual connection
        x = p_emb1 + pathway_feats
        x = x.view(num_samples, 258, -1)
        x = self.pre_transformer_ln(x)
        x = self.pre_transformer_proj(x)
        x_input = x
        transformer_attn_maps = []
        for layer in self.transformer_layers:
            x, attn = layer(x)
            transformer_attn_maps.append(attn)
        
        final_feat = x_input+x
        final_feat = self.token_dim_reduction(final_feat)
        
        # flatten
        final_feat = torch.flatten(final_feat, start_dim=1)
        out = self.net(final_feat)  #  [batch_size, num_class]

        return out,final_feat,gat_attn,edge_idx_attn,transformer_attn_maps,gene_atte1,gene_atte2
    


In [ ]:
# ====== Argument Parsing ====== #
parser = argparse.ArgumentParser()
parser.add_argument('--WORKDIR_PATH', type=str, default="Data/example_data")
parser.add_argument('--model_name', type=str, default='my')
parser.add_argument('--outdir',type=str,default="Data/example_data/test")

# === Train setting === #
parser.add_argument('--learning_rate', type=float, default=1e-4)
parser.add_argument('--epochs', type=int, default=100)
parser.add_argument('--batch_size', type=int, default=64)
parser.add_argument('--weight_decay', type=float, default=0)
parser.add_argument('--patience', type=int, default=10)
parser.add_argument('--testset_yes', type=bool, default=True)
args = parser.parse_known_args()[0]

In [ ]:
# Data loading
with open("Data/Pathway data/pathways_adjacency(20).pkl", "rb") as f:  
    pathways_matrix = pickle.load(f)
pathway_adj = pd.read_csv('Data/Pathway data/pathway_weight_adj(20).csv',index_col=0) 
exp_file  = 'Data/example_data/exp_data.csv'
exp_data = pd.read_csv(exp_file, index_col=0)
label_data = pd.read_csv("Data/example_data/label_data.csv")
pathway_order = list(pathways_matrix.keys())
pathway_adj = pathway_adj.loc[pathway_order, :]
pathway_adj = pathway_adj.loc[:, pathway_order]
sparseTensor = torch.tensor(pathway_adj.values).to_sparse().to(device)
pathway_ind = sparseTensor.indices()
pathway_weight = sparseTensor.values()


label_dict  = dict(zip(label_data['samples'], label_data['label']))
train_dataset = ExpressionDataset(exp_data, label_dict)
train_dataloader = DataLoader(train_dataset,batch_size=args.batch_size,shuffle=False,collate_fn=collate_fn)
other_exp = exp_data.shape[1] % args.batch_size
edge_index_b = []
for i in range(args.batch_size):
    offset = i * 258
    edge_index_b.append(pathway_ind + offset)
edge_index_b = torch.cat(edge_index_b, dim=1)
edge_weight_b = pathway_weight.repeat(args.batch_size).float()

other_train = exp_data.shape[1] % args.batch_size

edge_index_b_train = []
for i in range(other_exp):
    offset = i * 258
    edge_index_b_train.append(pathway_ind + offset)
edge_index_b_train = torch.cat(edge_index_b_train, dim=1)
edge_weight_b_train = pathway_weight.repeat(other_exp).float()


processor = GenePathwayProcessor(pathways_matrix)

In [ ]:

# Model initialization
model = PathwayGATModel(
    gene_in_dim=1, gene_hidden_dim1=32, gene_hidden_dim2=96,
    pathway_in_dim=128, pathway_hidden_dim1=128,
    MLP_input_dim=8256 , MLP_hidden_dim1=512, MLP_hidden_dim2=128,
    num_class=4, dropout_pathway=0.5, GAT_dropout=0.3,transformer_heads = 4,
    transformer_layers = 3,final_pathway_dim=32).to(device)

# Loss function
loss_fn = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam([
        {'params': model.parameters()},
        {'params': loss_fn.parameters()}
    ], lr=args.learning_rate, weight_decay=args.weight_decay) 

#model train
model, _, list_train_out,list_train_loss, list_train_true = train(
    model, exp_data, train_dataloader, edge_index_b,edge_index_b_train, edge_weight_b,edge_weight_b_train,args.batch_size,loss_fn, optimizer)